# Detecting small pill defects • single-seed evaluation
Do overlapping patches improve detection or localization on the official MVTec AD pill test set? We compare the two saved PyTorch autoencoders (training seed 42) with a frozen ImageNet feature baseline. The unit of evaluation is one original image.

Run all cells in order from the repository root or `notebooks/`. Install the repository requirements, place the official category under `data/pill/` (or set `MVTEC_PILL_DIR` to the category folder), and allow torchvision to cache its pretrained weights on first use. Outputs go to `outputs/evaluation/seed_42/`. The setup uses CPU inference for comparable timing across operating systems.

In [ ]:
import hashlib
import json
import math
import os
import platform
from pathlib import Path
from time import perf_counter

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import sklearn
    import torch
    import torchvision
    from IPython.display import display
    from PIL import Image
    from sklearn.metrics import (
        ConfusionMatrixDisplay, average_precision_score, confusion_matrix,
        f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
    )
    from torch import nn
    from torchvision import transforms
    from torchvision.models import ResNet18_Weights, resnet18
except (ImportError, OSError, RuntimeError) as exc:
    raise ImportError(
        "Install requirements.txt in this notebook's kernel. Use a compatible pair "
        "of torch and torchvision with the ResNet18_Weights API."
    ) from exc

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "split.csv").is_file() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open notebook 04 from this repository (split.csv is required).")
DATA = Path(os.environ.get("MVTEC_PILL_DIR", ROOT / "data" / "pill")).expanduser()
CHECKPOINTS = ROOT / "outputs" / "checkpoints"
OUT = ROOT / "outputs" / "evaluation" / "seed_42"
WHOLE, PATCH, CNN = "Whole-image AE", "Patch AE", "ResNet-18 1-NN"
METHODS = (WHOLE, PATCH, CNN)
SIZE = 256
AREA_BINS = [0, 0.01, 0.05, np.inf]
AREA_LABELS = ["<1%", "1% to <5%", "≥5%"]
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(min(4, torch.get_num_threads()))
resize_rgb = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])

def read_image(image_id):
    with Image.open(DATA / image_id) as image:
        if image.size != (800, 800):
            raise ValueError(f"Expected the original 800 × 800 image at {image_id}")
        return resize_rgb(image.convert("RGB"))

print(f"CPU • {torch.get_num_threads()} threads • PyTorch {torch.__version__} • {DATA}")

## Normal training and validation images
Reuse the committed assignment and check it against the original files. The feature bank uses only the 214 training IDs. Validation supplies the 53 scores for each method's operating threshold.

In [ ]:
if not (DATA / "train/good").is_dir():
    raise FileNotFoundError(f"Missing {DATA / 'train/good'}. Extract the official MVTec AD "
                            "pill category or set MVTEC_PILL_DIR and rerun setup.")
split = pd.read_csv(ROOT / "split.csv")
if set(split.columns) != {"relative_path", "split"}:
    raise ValueError("Expected relative_path and split columns in split.csv.")
listed = split.relative_path
on_disk = {p.relative_to(DATA).as_posix() for p in (DATA / "train/good").glob("*.png")}
if (listed.isna().any() or listed.duplicated().any() or set(listed) != on_disk
        or split.split.value_counts().to_dict() != {"train": 214, "val": 53}):
    raise ValueError("split.csv must partition all original train/good PNGs into 214 + 53 unique IDs.")
train_ids = sorted(split.loc[split.split == "train", "relative_path"])
val_ids = sorted(split.loc[split.split == "val", "relative_path"])
print(f"{len(train_ids)} reference images; {len(val_ids)} calibration images")

## Load the autoencoders for inference
Notebook 02's `ConvAutoencoder` and notebook 03's `PatchAutoencoder` use the same layer pattern with different depths. The constructor below reproduces their sequential layer names, kernel sizes, strides, padding, and activations. Strict loading checks every parameter. The whole-image file contains the state dictionary directly. The patch file wraps it with the inference geometry and background cutoffs.

In [ ]:
class ReconstructionAE(nn.Module):
    def __init__(self, channels):
        super().__init__()
        encoder, decoder = [], []
        for src, dst in zip(channels, channels[1:]):
            encoder.extend([nn.Conv2d(src, dst, 3, stride=2, padding=1), nn.ReLU()])
        for src, dst in zip(channels[:0:-1], channels[-2::-1]):
            decoder.extend([nn.ConvTranspose2d(src, dst, 4, stride=2, padding=1),
                            nn.Sigmoid() if dst == 3 else nn.ReLU()])
        self.encoder = nn.Sequential(*encoder)
        self.decoder = nn.Sequential(*decoder)

    def forward(self, image):
        return self.decoder(self.encoder(image))


def load_autoencoders(directory):
    def read_weights(filename):
        path = directory / filename
        if not path.is_file():
            raise FileNotFoundError(f"Required model checkpoint is missing: {path}")
        return torch.load(path, map_location="cpu", weights_only=True)

    whole = ReconstructionAE([3, 32, 64, 128, 256])
    whole.load_state_dict(read_weights("whole_image_autoencoder.pth"), strict=True)
    saved = read_weights("patch_autoencoder.pth")
    patch = ReconstructionAE([3, 32, 64, 128])
    patch.load_state_dict(saved["model_state"], strict=True)
    config = {k: saved[k] for k in ("image_size", "patch_size", "stride",
                                     "background_brightness", "background_spread")}
    if (config["image_size"], config["patch_size"], config["stride"]) != (SIZE, 128, 64):
        raise ValueError(f"Unexpected patch geometry in checkpoint: {config}")
    return whole.eval().requires_grad_(False), patch.eval().requires_grad_(False), config

whole, patch, config = load_autoencoders(CHECKPOINTS)
print(config)

## Reconstruction scores and coverage
Both models receive the same unaugmented RGB tensor in [0, 1]. A pixel's error is its mean absolute RGB residual. Following notebook 03, we accumulate patch **errors** at their original coordinates and divide by the number of retained patches covering each pixel. A filtered-out region with no coverage scores zero. The results retain the uncovered pixel count for every image. The image score averages the largest 656 of 65,536 pixel errors (rounding 1% up).

In [ ]:
def patch_windows(config):
    size, width, step = (config[k] for k in ("image_size", "patch_size", "stride"))
    starts = range(0, size - width + 1, step)
    return [(slice(y, y + width), slice(x, x + width)) for y in starts for x in starts]

windows = patch_windows(config)
full_cover = torch.zeros(SIZE, SIZE, dtype=torch.int32)
for region in windows:
    full_cover[region] += 1
if len(windows) != 9 or full_cover.min() != 1 or full_cover.max() != 4:
    raise ValueError("The unfiltered 3 × 3 grid must cover every pixel, with 1–4 contributions.")

@torch.inference_mode()
def whole_error(image):
    rebuilt = whole(image.unsqueeze(0))[0]
    return (rebuilt - image).abs().mean(dim=0)

@torch.inference_mode()
def patch_error(image):
    total = torch.zeros_like(image[0])
    cover = torch.zeros_like(full_cover)
    for region in windows:
        tile = image[:, region[0], region[1]]
        gray = tile.mean(dim=0)
        if gray.mean() < config["background_brightness"] and gray.std() < config["background_spread"]:
            continue
        rebuilt = patch(tile.unsqueeze(0))[0]
        total[region] += (rebuilt - tile).abs().mean(dim=0)
        cover[region] += 1
    return total / cover.clamp(min=1), int((cover == 0).sum())

def tail_score(heatmap):
    if heatmap.shape != (SIZE, SIZE):
        raise ValueError(f"Unexpected heatmap shape {heatmap.shape}")
    return heatmap.flatten().topk(math.ceil(0.01 * heatmap.numel())).values.mean().item()

## Frozen ImageNet baseline
Use torchvision ResNet-18 with the explicit `IMAGENET1K_V1` weights. Its weight preset takes the shared 256-pixel RGB tensor, center-crops to 224 × 224 and applies ImageNet channel normalization. Replacing the classifier with identity exposes the 512 global-average-pooled features. The score is the smallest Euclidean distance to any of the 214 raw training embeddings. Bank construction is a one-time setup cost.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
try:
    backbone = resnet18(weights=weights)
except (OSError, RuntimeError) as exc:
    raise RuntimeError(f"Cannot load the pretrained ResNet-18 weights. Allow a first-run "
                       f"download or populate the torchvision cache from {weights.url}.") from exc
backbone.fc = nn.Identity()
backbone.eval().requires_grad_(False)
preprocess_cnn = weights.transforms()

@torch.inference_mode()
def embedding(image):
    return backbone(preprocess_cnn(image).unsqueeze(0))[0]

bank_start = perf_counter()
bank = torch.stack([embedding(read_image(i)) for i in train_ids])
bank_seconds = perf_counter() - bank_start
if bank.shape != (214, 512) or not torch.isfinite(bank).all():
    raise ValueError("Expected 214 finite 512-dimensional reference embeddings.")
print(f"Reference bank {tuple(bank.shape)} built in {bank_seconds:.2f} s")

## Calibrate before opening the test set
Score the 53 normal validation images once per method. The threshold is the linearly interpolated 95th percentile, with a positive decision only for a score strictly above it. Warm-up uses a training image. Timings below measure synchronous, batch-one CPU inference from a shared decoded and resized tensor to the scalar score, including patch stitching or CNN preprocessing and bank lookup. Shared image I/O, plotting, and bank construction are excluded.

In [ ]:
@torch.inference_mode()
def infer(method, image):
    if method == CNN:
        distance = torch.linalg.vector_norm(bank - embedding(image), dim=1).min().item()
        return distance, None, None
    if method == WHOLE:
        heatmap, uncovered = whole_error(image), None
    elif method == PATCH:
        heatmap, uncovered = patch_error(image)
    else:
        raise ValueError(f"Unknown method {method}")
    return tail_score(heatmap), heatmap, uncovered

def score_images(image_ids, keep_maps=False):
    rows = []
    maps = {method: {} for method in (WHOLE, PATCH)}
    for index, image_id in enumerate(image_ids):
        image = read_image(image_id)
        # Rotate method order to distribute order effects in latency measurements.
        offset = index % len(METHODS)
        for method in METHODS[offset:] + METHODS[:offset]:
            start = perf_counter()
            score, heatmap, uncovered = infer(method, image)
            elapsed_ms = (perf_counter() - start) * 1000
            if not math.isfinite(score):
                raise ValueError(f"Non-finite score for {method} on {image_id}")
            rows.append(dict(method=method, image_id=image_id, score=score,
                             inference_ms=elapsed_ms, uncovered_pixels=uncovered))
            if keep_maps and heatmap is not None:
                maps[method][image_id] = heatmap.numpy()
    return pd.DataFrame(rows), maps

warmup = read_image(train_ids[0])
for method in METHODS:
    infer(method, warmup)
validation, _ = score_images(val_ids)
thresholds = validation.groupby("method").score.quantile(0.95, interpolation="linear")
display(thresholds.rename("validation_threshold").to_frame())

## Official test images and masks
Relative paths identify images uniquely across categories. Good images have an all-zero mask. Binary anomalous masks are resized to the error-map grid using nearest-neighbor interpolation. Defect area is the fraction of positive pixels in the **original** mask. The fixed area groups (<1%, 1% to <5%, ≥5%) describe the annotation footprint, including any change that spans much of the pill.

In [ ]:
def load_test_manifest():
    test_root = DATA / "test"
    if not test_root.is_dir():
        raise FileNotFoundError(f"Missing official test directory: {test_root}")
    categories = {"color", "combined", "contamination", "crack", "faulty_imprint",
                  "pill_type", "scratch", "good"}
    if {p.name for p in test_root.iterdir() if p.is_dir()} != categories:
        raise ValueError("Expected good and all seven original MVTec pill test directories.")
    rows, masks = [], {}
    for path in sorted(test_root.glob("*/*.png")):
        image_id = path.relative_to(DATA).as_posix()
        category = path.parent.name
        label = int(category != "good")
        area = 0.0
        if label:
            mask_path = DATA / "ground_truth" / category / f"{path.stem}_mask.png"
            if not mask_path.is_file():
                raise FileNotFoundError(f"Missing ground truth for {image_id}: {mask_path}")
            with Image.open(mask_path) as mask:
                if mask.size != (800, 800):
                    raise ValueError(f"Unexpected mask geometry: {mask_path}")
                binary = np.asarray(mask.convert("L")) > 0
                area = binary.mean()
                if not area:
                    raise ValueError(f"Empty anomaly mask: {mask_path}")
                resized = Image.fromarray(binary).resize((SIZE, SIZE), Image.Resampling.NEAREST)
                masks[image_id] = np.asarray(resized, dtype=bool)
        else:
            masks[image_id] = np.zeros((SIZE, SIZE), dtype=bool)
        rows.append(dict(image_id=image_id, category=category, label=label,
                         defect_area_fraction=area, defect_pixels_256=int(masks[image_id].sum())))
    manifest = pd.DataFrame(rows)
    if (len(manifest) != 167 or set(manifest.category) != categories
            or manifest.label.value_counts().to_dict() != {1: 141, 0: 26}):
        raise ValueError("Expected 26 good and 141 anomalous official test images in 7 defect groups.")
    return manifest, masks

manifest, masks = load_test_manifest()
display(manifest.groupby("category").size().rename("test_images").to_frame())

In [ ]:
test_scores, maps = score_images(manifest.image_id, keep_maps=True)
predictions = test_scores.merge(manifest, on="image_id", validate="many_to_one")
predictions["threshold"] = predictions.method.map(thresholds)
predictions["prediction"] = (predictions.score > predictions.threshold).astype(int)
predictions["area_group"] = pd.cut(predictions.defect_area_fraction, AREA_BINS,
                                   labels=AREA_LABELS, right=False)
predictions.loc[predictions.label == 0, "area_group"] = pd.NA

coverage = pd.concat([validation.assign(split="val"), predictions.assign(split="test")])
coverage = coverage.loc[coverage.method == PATCH].groupby("split").uncovered_pixels
display(coverage.agg(images="size", max_uncovered="max",
                     partially_or_fully_uncovered=lambda s: (s > 0).sum()))

## Detection, localization, and runtime
Image ROC-AUC uses continuous scores. Precision, recall, F1, and confusion counts use the fixed validation threshold. Pixel AP pools every pixel in all 167 test images, including good images, before ranking each autoencoder's continuous errors. It has no pixel threshold. Subgroup recall uses only anomalous images in that group. Empty groups retain a zero count and undefined recall.

In [ ]:
mask_vector = np.stack([masks[i] for i in manifest.image_id]).ravel()
summary_rows = []
for method in METHODS:
    group = predictions.loc[predictions.method == method]
    tn, fp, fn, tp = confusion_matrix(group.label, group.prediction, labels=[0, 1]).ravel()
    pixel_ap = np.nan
    if method in maps:
        pixel_scores = np.stack([maps[method][i] for i in manifest.image_id]).ravel()
        pixel_ap = average_precision_score(mask_vector, pixel_scores)
        del pixel_scores
    summary_rows.append(dict(method=method, test_n=len(group), threshold=thresholds[method],
                             roc_auc=roc_auc_score(group.label, group.score),
                             precision=precision_score(group.label, group.prediction, zero_division=0),
                             recall=recall_score(group.label, group.prediction),
                             f1=f1_score(group.label, group.prediction), tn=tn, fp=fp, fn=fn, tp=tp,
                             normal_n=tn + fp, false_positive_rate=fp / (tn + fp), pixel_ap=pixel_ap))
overall = pd.DataFrame(summary_rows).set_index("method")
anomalies = predictions.loc[predictions.label == 1]

def subgroup_results(column):
    counts = anomalies.groupby(["method", column], observed=False).prediction.agg(
        sample_n="size", detected="sum")
    counts["missed"] = counts.sample_n - counts.detected
    counts["recall"] = counts.detected.div(counts.sample_n.replace(0, np.nan))
    return counts

by_type = subgroup_results("category")
by_area = subgroup_results("area_group")
timing = predictions.groupby("method").inference_ms.agg(
    images="size", mean_ms="mean", median_ms="median", p95_ms=lambda s: s.quantile(.95))
display(overall.round(4), by_type, by_area, timing.round(2))

## Comparison figures and machine-readable results
For qualitative review, select the first image in filename order for each available AE decision pattern (both detected, only one detected, both missed), plus a normal false alarm if present. These are diagnostic examples; the count tables describe prevalence. Both heatmap columns share the same color scale across all panels. The masks show the target at model resolution.

In [ ]:
fig_roc, ax = plt.subplots(figsize=(6, 5), layout="constrained")
fig_cm, axes = plt.subplots(1, 3, figsize=(10, 3.4), layout="constrained")
for method, cm_ax in zip(METHODS, axes):
    group = predictions.loc[predictions.method == method]
    fpr, tpr, _ = roc_curve(group.label, group.score)
    ax.plot(fpr, tpr, label=f"{method} ({overall.loc[method, 'roc_auc']:.3f})")
    ConfusionMatrixDisplay.from_predictions(group.label, group.prediction, labels=[0, 1],
        display_labels=["Good", "Defect"], ax=cm_ax, colorbar=False, cmap="Blues")
    cm_ax.set_title(method)
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set(xlabel="False-positive rate", ylabel="True-positive rate", title="Official test set ROC")
ax.legend()
fig_time, ax = plt.subplots(figsize=(6, 3.5), layout="constrained")
ax.bar(METHODS, timing.loc[list(METHODS), "median_ms"])
ax.set(ylabel="Median milliseconds per image", title="Single-image CPU inference (167 test images)")
plt.show()

In [ ]:
decisions = predictions.pivot(index="image_id", columns="method", values="prediction")
decisions = decisions.join(manifest.set_index("image_id").label).sort_index()
defective = decisions.label == 1
patterns = {
    "Both detected": defective & (decisions[WHOLE] == 1) & (decisions[PATCH] == 1),
    "Whole only": defective & (decisions[WHOLE] == 1) & (decisions[PATCH] == 0),
    "Patch only": defective & (decisions[WHOLE] == 0) & (decisions[PATCH] == 1),
    "Both missed": defective & (decisions[WHOLE] == 0) & (decisions[PATCH] == 0),
    "Normal false alarm": ~defective & ((decisions[WHOLE] == 1) | (decisions[PATCH] == 1)),
}
examples = [(name, decisions.index[rows][0]) for name, rows in patterns.items() if rows.any()]
vmax = max(float(maps[m][i].max()) for _, i in examples for m in (WHOLE, PATCH))
fig_maps, axes = plt.subplots(len(examples), 4, squeeze=False, figsize=(10, 2.7 * len(examples)),
                              layout="constrained")
for row, (pattern, image_id) in enumerate(examples):
    axes[row, 0].imshow(read_image(image_id).permute(1, 2, 0))
    axes[row, 0].set_title(f"{pattern}\n{image_id}", fontsize=9)
    axes[row, 1].imshow(masks[image_id], cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Ground truth")
    for col, method in enumerate((WHOLE, PATCH), start=2):
        heat = axes[row, col].imshow(maps[method][image_id], cmap="inferno", vmin=0, vmax=vmax)
        tag = "defect" if decisions.loc[image_id, method] else "good"
        axes[row, col].set_title(f"{method} • {tag}")
    for ax in axes[row]:
        ax.axis("off")
fig_maps.colorbar(heat, ax=axes[:, 2:].ravel().tolist(), shrink=.6, label="Mean absolute RGB error")
plt.show()

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
for filename, table in [("overall", overall), ("per_defect_type", by_type),
                        ("per_defect_area", by_area), ("inference_time", timing)]:
    table.to_csv(OUT / f"{filename}.csv")
predictions.to_csv(OUT / "test_predictions.csv", index=False)
validation.assign(threshold=validation.method.map(thresholds)).to_csv(
    OUT / "normal_validation_scores.csv", index=False)
for filename, fig in [("roc_curves", fig_roc), ("confusion_matrices", fig_cm),
                      ("inference_time", fig_time), ("localization_examples", fig_maps)]:
    fig.savefig(OUT / f"{filename}.png", dpi=150, bbox_inches="tight")

files = [ROOT / "split.csv", CHECKPOINTS / "whole_image_autoencoder.pth",
         CHECKPOINTS / "patch_autoencoder.pth"]
metadata = dict(training_seed=42, torch=str(torch.__version__), torchvision=torchvision.__version__,
                sklearn=sklearn.__version__, python=platform.python_version(), device="CPU",
                processor=platform.processor(), threads=torch.get_num_threads(),
                cnn_weights=str(weights), bank_build_seconds=bank_seconds, patch_config=config,
                sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in files})
(OUT / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Evaluation tables, predictions, figures, and run metadata saved to {OUT}")